# DAPI_PML_SUMO_3D

This notebook performs the following analysis in 3D:
- Detect nuclei
- For each nucleus:
    - Detect PML foci and compute the following parameters for each focus: volume, mean and sum intensity in PML channel, mean and sum intensity in SUMO2 channel
    - Measure SUMO2 mean and sum intensity outside the PML foci
  

**Authors:** Thomas Caille & Héloïse Monnet, ORION image analysis facility

**Source code and additional information:** <https://github.com/orion-cirb/DAPI_PML_SUMO> (access restricted - requires GitHub account with granted permissions)

## 1. Import libraries

In [1]:
from aicsimageio import AICSImage
from cellpose import models
import datetime
import logging
import matplotlib.pyplot as plt
import napari
import numpy as np
import os
import pandas as pd
import scipy as sc
import skimage as sk
import tifffile as tf

## 2. Enter parameters

In [2]:
# Directory containing the input images for processing (must end with '/')
img_dir = '/home/heloise/Images/De Thé/Adèle/DAPI PML SUMO2 /Images 3D - 63x zoom2 - marquage 2 - C1 PML C2 SUMO/'

# Directory name where results will be saved (must end with '/')
result_dir_name = 'results/'

# Channel indices in the image data (1-based indexing)
channel_DAPI = 3                # Channel index for Nuclei marker
channel_PML = 1                 # Channel index for PML marker
channel_SUMO = 2                # Channel index for SUMO marker

# Parameters for the 2D detection of nuclei with Cellpose 3
cellpose_model = 'nuclei'       # Cellpose model name
cellpose_diameter = 200         # Estimated nucleus diameter in pixels

# Area constraints to filter detected nuclei
nucleus_area_min = 50       # Minimum allowed nucleus area (in µm²)
nucleus_area_max = 300      # Maximum allowed nucleus area (in µm²)

# Parameters for the 3D detection of PML foci
foci_dog1 = 1               # Difference of Gaussians filter sigma 1 (sigma 1 < sigma 2)
foci_dog2 = 3               # Difference of Gaussians filter sigma 2 (sigma 1 < sigma 2)

# Volume constraints to filter detected PML foci
foci_vol_min = 0.005      # Minimum allowed foci volume (in µm³)
foci_vol_max = 4          # Maximum allowed foci volume (in µm³)

## 3. Declare utility functions

In [3]:
def open_img_czi(img_dir, img_name, verbose=False):
    '''
    Open a .czi image file and extract voxel size from metadata 

    Parameters:
        img_dir (str): Directory containing the .czi image
        img_name (str): Name of the image file
        verbose (bool): If True, prints voxel size and image dimensions for inspection. Defaults to False.

    Returns:
        img (ndarray): 4D image array (C, Z, Y, X)
        voxel_size (list of float): [Z, Y, X] voxel dimensions in µm
    '''
    czi = AICSImage(img_dir+img_name)
    img = czi.get_image_data("CZYX", T=0)
    voxel_size = [czi.physical_pixel_sizes.Z, czi.physical_pixel_sizes.Y, czi.physical_pixel_sizes.X]

    if verbose:
        print('Voxel size (ZYX):', voxel_size, 'µm')
        print('Image size (CZYX):', img.shape)

    return img, voxel_size

def get_napari_viewer():
    """
    Launch or retrieve the current Napari viewer in 3D mode.

    Returns:
        napari.Viewer: Active or newly created Napari viewer.
    """
    # Check if a viewer is already running
    viewer = napari.current_viewer()
    
    if viewer is None:
        # Create a new viewer
        viewer = napari.Viewer()
        napari.run()
    else:
        # Remove all existing layers
        viewer.layers.clear()

    viewer.dims.ndisplay = 3
    return viewer

def cellpose_detection(img, model, diameter, cellprob_threshold, flow_threshold):
    """
    Run 2D Cellpose segmentation on the input image.

    Parameters:
        img (ndarray): 2D image (YX)
        model (str): Cellpose model name
        diameter (float): Estimated object diameter in pixels
        cellprob_threshold (float): Minimum probability value for a pixel to be considered part of a cell.
        flow_threshold (float): Threshold on the consistency of the spatial flow field used to reconstruct masks.

    Returns:
        masks (ndarray): 2D label image with segmented objects
    """
    model = models.CellposeModel(gpu=True, pretrained_model=model)
    masks, _, _ = model.eval(img,
                            diameter=diameter,
                            cellprob_threshold=cellprob_threshold,
                            flow_threshold=flow_threshold)
    return masks

def filter_labels_size(label_img, min_voxels=0, max_voxels=np.inf):
    """
    Filter labeled regions by their size (in voxels).

    Parameters:
        label_img (ndarray): 3D label image (ZYX)
        min_voxels (int): Minimum voxel count to keep a region
        max_voxels (int or float): Maximum voxel count to keep a region

    Returns:
        filtered (ndarray): 3D label image with size-filtered regions relabeled
    """
    valid_labels = [
        r.label for r in sk.measure.regionprops(label_img)
        if min_voxels <= r.area <= max_voxels
    ]
    filtered = np.isin(label_img, valid_labels) * label_img
    return sk.segmentation.relabel_sequential(filtered)[0]

def compute_background(img, mask_objects, focused_slices_id):
    """
    Computes the background noise intensity outside segmented objects 
    within a specified range of focused slices in a 3D image stack.
    
    Parameters:
        img (ndarray): 3D image (ZYX)
        mask_objects (ndarray): 3D binary mask of objects (ZYX)
        focused_slices_id: List of indices of focused slices
        
    Returns:
        float: Background noise intensity
    """
     # Create a background mask (1 = background, 0 = object)
    mask_bg = sk.util.invert(mask_objects).astype(np.uint8)
    # Restrict background mask to only the focused slice range
    mask_bg[:focused_slices_id[0]] = 0
    mask_bg[focused_slices_id[-1]+1:] = 0
    
    # Extract background pixel intensities and compute the median
    background = np.median(img[np.nonzero(mask_bg)])
    return background

def maximum_intensity_projection(img) :
    """
    Computes the maximum intensity projection of a 3D image stack.
    
    Parameters:
        img (ndarray): 3D image (ZYX)
        
    Returns:
        ndarray: 2D image (YX)
    """
    img_max = np.max(img, axis=0)
    return img_max

def laplacian_variance(img):
    """
    Compute the variance of the Laplacian of a 2D image.
    
    Parameters:
        img (ndarray): 2D image (YX)
        
    Returns:
        float: Variance of the Laplacian of the image
    """
    kernel = np.array([[0,  1, 0],
                       [1, -4, 1],
                       [0,  1, 0]])
    lap = sc.ndimage.convolve(img.astype(float), kernel)
    return np.var(lap)
    
Image = np.ndarray

def find_focused_slices_id(img, max_var_threshold):
    """
    Find indices of slices with high focus based on Laplacian variance.
    
    Parameters:
        img (ndarray): 3D image (ZYX)
        max_var_threshold (float): Threshold coefficient between 0 and 1. A slice is selected if its Laplacian variance 
                                    is greater than (max_var_threshold * max_variance).
        
    Returns:
        ndarray: Indices of slices whose variance passes the threshold.
    """
    lap_vars = []
    for slice in img:
        lap_vars.append(laplacian_variance(slice))
    max_var = max(lap_vars)
    indices = np.where(lap_vars > max_var_threshold*max_var)[0]
    print('Focused slices:', indices)
    return indices

## 4. Load images

In [4]:
# List all files in the image directory with '.czi' file format
img_names = [item for item in os.listdir(img_dir) if item.endswith('czi')]
img_names.sort()
print(f"Found {len(img_names)} images to analyze in directory: {img_dir}")

# Create results folder
result_dir = img_dir + result_dir_name
os.makedirs(result_dir, exist_ok=True)
print(f"\nResults directory successfully created at: {result_dir}")

Found 12 images to analyze in directory: /home/heloise/Images/De Thé/Adèle/DAPI PML SUMO2 /Images 3D - 63x zoom2 - marquage 2 - C1 PML C2 SUMO/

Results directory successfully created at: /home/heloise/Images/De Thé/Adèle/DAPI PML SUMO2 /Images 3D - 63x zoom2 - marquage 2 - C1 PML C2 SUMO/results/


### Preview the image of your choice

In [5]:
# Open image
img_name = img_names[0]
print('Image name:', img_name)
img, voxel_size = open_img_czi(img_dir, img_name, True)
        
# Display image in Napari
viewer = get_napari_viewer()
viewer.add_image(img, name="image", scale=voxel_size)

Image name: C217Vn2 Mp2 1h DA-63x-1.czi
Voxel size (ZYX): [0.25, 0.05501991016313937, 0.05501991016313937] µm
Image size (CZYX): (3, 20, 1024, 1024)


<Image layer 'image' at 0x73b8c1a95690>

## 5. Analyze images

In [6]:
start_time = datetime.datetime.now()
logging.basicConfig(level=logging.WARN, force=True)

# Initialize list to collect per-image dataframe
df_list = []

# Loop through each image file
for index, img_name in enumerate(img_names, start=1):
    print(f"\nAnalyzing image: {img_name} ({index}/{len(img_names)})")

    # Load image and voxel size from metadata
    img, voxel_size = open_img_czi(img_dir, img_name)
    voxel_vol = np.prod(voxel_size)

    # Extract various channels
    img_DAPI = img[channel_DAPI-1]
    img_PML = img[channel_PML-1]
    img_SUMO = img[channel_SUMO-1]

    # Identify focused slices on PML channel
    focused_slices_id = find_focused_slices_id(img_PML, 0.4)

    # --- DETECT NUCLEI IN 2D ---
    # Perform a maximum intensity projection
    img_DAPI_mip = maximum_intensity_projection(img_DAPI)
    # Run Cellpose detection
    nuclei = cellpose_detection(img_DAPI_mip, cellpose_model, cellpose_diameter, 0.0, 0.4)
    print(f"Nuclei detected: {np.max(nuclei)}")
    # Filter masks by volume
    voxel_area_xy = voxel_size[1]*voxel_size[2]
    nuclei_labels = filter_labels_size(nuclei, 
                                         min_voxels=nucleus_area_min / voxel_area_xy,
                                         max_voxels=nucleus_area_max / voxel_area_xy)
    print(f"Nuclei remaining after filtering: {np.max(nuclei_labels)}")
    
    # --- EXTEND NUCLEI MASK INTO 3D --
    # Extend 2D nuclei mask across the Z dimension
    nuclei_labels = nuclei_labels[np.newaxis, :, :]  # shape -> (1, Y, X)
    nuclei_labels = np.repeat(nuclei_labels, img_DAPI.shape[0], axis=0) # shape -> (Z, Y, X)
    # Keep nuclei mask only within the focused slices
    nuclei_labels[:focused_slices_id[0]] = 0
    nuclei_labels[focused_slices_id[-1]+1:] = 0
    # Create nuclei binary mask
    nuclei_binary = nuclei_labels > 0
    # Save result as .tif file
    tf.imwrite(result_dir + img_name.replace(".czi", "_nuclei.tif"), nuclei_labels, imagej=True, resolution=(1/voxel_size[1], 1/voxel_size[2]), metadata={'spacing': voxel_size[0], 'unit': 'um', 'axes': 'ZYX'})
    
    # --- DETECT PML FOCI IN 3D ---
    # Preprocessing: Difference of Gaussian filter
    foci_binary = sk.filters.difference_of_gaussians(img_PML, foci_dog1, foci_dog2)
    # Thresholding: Triangle method
    thr = sk.filters.threshold_triangle(foci_binary)
    foci_binary = foci_binary > thr
    # Postprocessing: 3D opening filter
    foci_binary = sk.morphology.binary_opening(foci_binary, footprint=sk.morphology.ball(2))
    # Remove foci outside nuclei
    foci_binary = foci_binary * nuclei_binary
    # Label objects in 3D
    foci_labels = sk.morphology.label(foci_binary, connectivity=1).astype(np.uint16)
    print(f"Foci detected: {np.max(foci_labels)}")
    # Filter objects by volume
    foci_labels = filter_labels_size(foci_labels,
                                        min_voxels=foci_vol_min / voxel_vol,
                                        max_voxels=foci_vol_max / voxel_vol)
    print(f"Foci detected after filtering: {np.max(foci_labels)}")
    # Create foci binary mask
    foci_binary = foci_labels > 0
    # Save result as .tif file
    tf.imwrite(result_dir + img_name.replace(".czi", "_foci.tif"), foci_labels, imagej=True, resolution=(1/voxel_size[1], 1/voxel_size[2]), metadata={'spacing': voxel_size[0], 'unit': 'um', 'axes': 'ZYX'})
    
    # --- NORMALIZE PML & SUMO CHANNELS ---
    # Compute PML and SUMO background noise
    bg_PML = compute_background(img_PML, nuclei_binary, focused_slices_id)
    bg_SUMO = compute_background(img_SUMO, nuclei_binary, focused_slices_id)
    print(f"PML background noise: {bg_PML:.3f}")
    print(f"SUMO background noise: {bg_SUMO:.3f}")
    # Substract them to the original channel
    img_PML_norm = np.clip(img_PML.astype(np.float32) - bg_PML, a_min=0, a_max=None).astype(np.uint16)
    img_SUMO_norm = np.clip(img_SUMO.astype(np.float32) - bg_SUMO, a_min=0, a_max=None).astype(np.uint16)

    # --- EXTRACT MEASUREMENTS ---
    # Compute per-foci measurements (volume, PML mean and sum intensity)
    df = pd.DataFrame(sk.measure.regionprops_table(foci_labels, 
                                                   intensity_image=img_PML_norm,
                                                   properties=['label', 'area', 'intensity_mean'],
                                                   spacing=voxel_size))
    df = df.rename(columns={'label': 'foci_label', 'area': 'foci_vol_um', 'intensity_mean': 'foci_pml_int_mean_bgcorr'})
    df['foci_pml_int_sum_bgcorr'] = df['foci_pml_int_mean_bgcorr'] * df['foci_vol_um'] / voxel_vol

    # Add per-foci SUMO mean and sum intensity measurements
    df['foci_sumo_int_mean_bgcorr'] = sk.measure.regionprops_table(foci_labels, 
                                                                   intensity_image=img_SUMO_norm,
                                                                   properties=['intensity_mean'],
                                                                   spacing=voxel_size)['intensity_mean']    
    df['foci_sumo_int_sum_bgcorr'] = df['foci_sumo_int_mean_bgcorr'] * df['foci_vol_um'] / voxel_vol

    # Map each foci to its corresponding nucleus
    df['nucleus_label'] = sk.measure.regionprops_table(foci_labels, 
                                                       intensity_image=nuclei_labels,
                                                       properties=['intensity_max'],
                                                       spacing=voxel_size)['intensity_max'].astype(int)

    # Add per-nucleus volume
    df_nuclei = pd.DataFrame(sk.measure.regionprops_table(nuclei_labels,
                                                          properties=['label', 'area'],
                                                          spacing=voxel_size))
    df_nuclei = df_nuclei.rename(columns={'area': 'nucleus_vol_um'})
    df = df.join(df_nuclei.set_index('label'), on='nucleus_label')

    # Add per-nucleus volume, SUMO mean and sum intensity outside foci
    foci_binary_inv = sk.util.invert(foci_binary)
    nuclei_labels_nofoci = nuclei_labels * foci_binary_inv
    df_nuclei_nofoci = pd.DataFrame(sk.measure.regionprops_table(nuclei_labels_nofoci, 
                                                                 intensity_image=img_SUMO_norm,
                                                                 properties=['label','area','intensity_mean'],
                                                                 spacing=voxel_size))
    df_nuclei_nofoci = df_nuclei_nofoci.rename(columns={'area': 'nucleus_nofoci_vol_um', 'intensity_mean': 'nucleus_nofoci_sumo_int_mean_bgcorr'})
    df_nuclei_nofoci['nucleus_nofoci_sumo_int_sum_bgcorr'] = df_nuclei_nofoci['nucleus_nofoci_vol_um'] * df_nuclei_nofoci['nucleus_nofoci_sumo_int_mean_bgcorr'] / voxel_vol
    df = df.join(df_nuclei_nofoci.set_index('label'), on='nucleus_label')

    # Add metadata columns (image name, number of focused slices, PML and SUMO background noise)
    df.insert(0, 'image', img_name)
    df.insert(1, 'nb_focused_slices', focused_slices_id[-1]-focused_slices_id[0]+1)
    df.insert(2, 'bg_pml', bg_PML)
    df.insert(3, 'bg_sumo', bg_SUMO)
    
    # Sort results by nucleus → foci hierarchy
    df = df.sort_values(by=['nucleus_label', 'foci_label'])

    # Append dataframe to the list for aggregation across images
    df_list.append(df)

# Mege all per-image dataframes into a single table and save it as CSV   
df_final = pd.concat(df_list, ignore_index=True)
df_final.to_csv(result_dir+'results.csv', index=False, float_format='%.3f')

print(f"\nAnalysis done!")
print(f"Execution time: {datetime.datetime.now() - start_time}")


Analyzing image: C217Vn2 Mp2 1h DA-63x-1.czi (1/12)
Focused slices: [ 7  8  9 10 11 12 13 14]
Nuclei detected: 18
Nuclei remaining after filtering: 15
Foci detected: 332
Foci detected after filtering: 332
PML background noise: 54.000
SUMO background noise: 354.000

Analyzing image: C217Vn2 Mp2 1h DA-63x-2.czi (2/12)
Focused slices: [ 7  8  9 10 11 12 13 14]
Nuclei detected: 18
Nuclei remaining after filtering: 13
Foci detected: 454
Foci detected after filtering: 453
PML background noise: 56.000
SUMO background noise: 364.000

Analyzing image: C217Vn2 Mp2 2h DA-63x-1.czi (3/12)
Focused slices: [ 6  7  8  9 10 11 12]
Nuclei detected: 24
Nuclei remaining after filtering: 21
Foci detected: 294
Foci detected after filtering: 291
PML background noise: 52.000
SUMO background noise: 504.000

Analyzing image: C217Vn2 Mp2 UT DA-63x-1.czi (4/12)
Focused slices: [ 7  8  9 10 11 12 13]
Nuclei detected: 12
Nuclei remaining after filtering: 11
Foci detected: 166
Foci detected after filtering: 165
PM

### Check detections for the image of your choice

In [7]:
# Open image
img_name = img_names[0]
print('Image name:', img_name)
img, voxel_size = open_img_czi(img_dir, img_name)

# Open corresponding nuclei and foci segmentation masks
nuclei = sk.io.imread(result_dir + img_name.replace(".czi", "_nuclei.tif"))
print(f"Nuclei number: {np.max(nuclei)}")
foci = sk.io.imread(result_dir + img_name.replace(".czi", "_foci.tif"))
print(f"Foci number: {np.max(foci)}")
        
# Display image and masks overlay in Napari
viewer = get_napari_viewer()
viewer.add_image(img, name="image", scale=voxel_size)
viewer.add_labels(nuclei, opacity=0.5, scale=voxel_size)
viewer.add_labels(foci, opacity=0.5, scale=voxel_size)

Image name: C217Vn2 Mp2 1h DA-63x-1.czi
Nuclei number: 15
Foci number: 332


<Labels layer 'foci' at 0x73b690393160>

### Check results file

In [8]:
df = pd.read_csv(result_dir+'results.csv')
df.head()

,image,nb_focused_slices,bg_pml,bg_sumo,foci_label,foci_vol_um,foci_pml_int_mean_bgcorr,foci_pml_int_sum_bgcorr,foci_sumo_int_mean_bgcorr,foci_sumo_int_sum_bgcorr,nucleus_label,nucleus_vol_um,nucleus_nofoci_vol_um,nucleus_nofoci_sumo_int_mean_bgcorr,nucleus_nofoci_sumo_int_sum_bgcorr
0,C217Vn2 Mp2 1h DA-63x-1.czi,8,54.0,354.0,1,0.166,5573.991,1226278.0,237.877,52333.0,1,170.037,164.785,748.963,163079183.0
1,C217Vn2 Mp2 1h DA-63x-1.czi,8,54.0,354.0,2,0.602,10388.930,8259199.0,557.639,443323.0,1,170.037,164.785,748.963,163079183.0
2,C217Vn2 Mp2 1h DA-63x-1.czi,8,54.0,354.0,3,0.360,12823.248,6103866.0,605.315,288130.0,1,170.037,164.785,748.963,163079183.0
3,C217Vn2 Mp2 1h DA-63x-1.czi,8,54.0,354.0,4,0.069,2688.648,244667.0,2503.044,227777.0,1,170.037,164.785,748.963,163079183.0
4,C217Vn2 Mp2 1h DA-63x-1.czi,8,54.0,354.0,5,0.336,7621.887,3384118.0,2951.892,1310640.0,1,170.037,164.785,748.963,163079183.0


In [9]:
df.describe()

,nb_focused_slices,bg_pml,bg_sumo,foci_label,foci_vol_um,foci_pml_int_mean_bgcorr,foci_pml_int_sum_bgcorr,foci_sumo_int_mean_bgcorr,foci_sumo_int_sum_bgcorr,nucleus_label,nucleus_vol_um,nucleus_nofoci_vol_um,nucleus_nofoci_sumo_int_mean_bgcorr,nucleus_nofoci_sumo_int_sum_bgcorr
count,3287.000000,3287.000000,3287.000000,3287.000000,3287.000000,3287.000000,3.287000e+03,3287.000000,3.287000e+03,3287.000000,3287.000000,3287.000000,3287.000000,3.287000e+03
mean,8.304837,56.459690,366.133861,156.241253,0.224231,4307.419457,2.058911e+06,2480.568054,9.790315e+05,7.678734,251.696278,245.346622,665.542984,2.004284e+08
std,1.870142,2.122712,54.577677,103.023306,0.208482,4814.455317,5.302615e+06,3931.970066,2.379760e+06,4.783911,96.777937,94.764717,317.980222,8.798363e+07
min,4.000000,52.000000,296.000000,1.000000,0.008000,699.781000,1.820100e+04,46.606000,1.538000e+03,1.000000,50.179000,49.352000,154.180000,1.825943e+07
25%,7.000000,55.000000,343.000000,70.000000,0.092000,1747.132000,2.292765e+05,592.103500,9.632300e+04,4.000000,182.564000,175.047000,419.395000,1.343265e+08
50%,8.000000,57.000000,354.000000,144.000000,0.169000,2832.777000,6.086250e+05,1164.910000,2.362250e+05,7.000000,229.269000,223.641000,616.391000,1.895453e+08
75%,8.000000,58.000000,367.000000,230.000000,0.285000,4805.733000,1.615564e+06,2374.549000,6.660800e+05,11.000000,300.479000,293.059000,882.682000,2.575532e+08
max,12.000000,59.000000,539.000000,453.000000,2.746000,43149.322000,6.839517e+07,41486.870000,3.060586e+07,21.000000,579.550000,568.317000,2242.109000,5.120826e+08
